# DD Startup Analysis - Parallel Coordinates Plot

This notebook reads HDF5 results from DD startup simulations and creates an interactive parallel coordinates plot showing the relationship between input parameters and economic outcomes.

## Objective
- Load simulation results from HDF5 files
- Filter to show only input parameters (excluding calculated variables)
- Create parallel coordinates plot with Dollar_Lost as color scale
- Visualize parameter sensitivities and trade-offs

## Configuration Options

Set your analysis parameters here before running the notebook:

In [ ]:
# =============================================================================
# CONFIGURATION OPTIONS - Set these before running the analysis
# =============================================================================

# File selection - Choose which HDF5 file to analyze
SELECTED_FILE = "dd_startup_results_20250916_161036.h5"  # Set to None for automatic selection, or specify filename
# Examples:
# SELECTED_FILE = 'dd_startup_results_20250916_120903.h5'
# SELECTED_FILE = 'dd_startup_results_small_test_20250916_123548.h5'

# File selection preferences (used when SELECTED_FILE = None):
PREFER_SMALL_TEST = False    # True: prefer small test files, False: prefer full datasets
USE_MOST_RECENT = True       # True: use most recent file if multiple matches

# Target variable selection - Choose which output metric to visualize
TARGET_VARIABLE = 'Dollar_Lost'    # Primary target variable for coloring

# Alternative target variables you can use:
# TARGET_VARIABLE = 't_startup'           # Startup time [s]
# TARGET_VARIABLE = 'Q_DD_total'          # Q factor for DD phase
# TARGET_VARIABLE = 'Q_DT_full_total'     # Q factor for DT phase
# TARGET_VARIABLE = 'P_fusion_DD_avg'     # Average DD fusion power [MW]
# TARGET_VARIABLE = 'P_e_net_DD_avg'      # Average DD net electric power [MW]
# TARGET_VARIABLE = 'E_fusion_total_DD'   # Total DD fusion energy [MJ]
# TARGET_VARIABLE = 'E_e_net_DD'          # Net DD electric energy [MJ]
# TARGET_VARIABLE = 'n_T_final'           # Final tritium density [m⁻³]

# Economic filter (only used when TARGET_VARIABLE = 'Dollar_Lost')
ECONOMIC_FILTER = 5e6        # Set to dollar amount (e.g., 1e8) or None

# Metric filter (used for non-economic targets)
METRIC_FILTER = None         # Set to threshold value or None
                            # Examples: 
                            # For t_startup: 1000 (seconds)
                            # For Q_DD_total: 0.1 (dimensionless)
                            # For power: 50 (MW)

# Color and styling options
USE_DISCRETE_COLORS = True   # True for discrete chunks, False for gradient
N_COLOR_CHUNKS = 6           # Number of discrete color levels

# Auto-determine appropriate filter type based on target variable
if TARGET_VARIABLE == 'Dollar_Lost':
    FILTER_VALUE = ECONOMIC_FILTER
    FILTER_TYPE = 'economic'
else:
    FILTER_VALUE = METRIC_FILTER
    FILTER_TYPE = 'metric'

print(f"🎯 Configuration Summary:")
print(f"   File Selection: {SELECTED_FILE if SELECTED_FILE else 'Automatic'}")
if SELECTED_FILE is None:
    print(f"   Preferences: {'Small test files' if PREFER_SMALL_TEST else 'Full datasets'}, {'Most recent' if USE_MOST_RECENT else 'First found'}")
print(f"   Target Variable: {TARGET_VARIABLE}")
print(f"   Filter Type: {FILTER_TYPE}")
print(f"   Filter Value: {FILTER_VALUE}")
print(f"   Color Style: {'Discrete chunks' if USE_DISCRETE_COLORS else 'Continuous gradient'}")
print(f"   Color Levels: {N_COLOR_CHUNKS}")

🎯 Configuration Summary:
   File Selection: dd_startup_results_test_20250916_132839.h5
   Target Variable: Dollar_Lost
   Filter Type: economic
   Filter Value: 1000000000.0
   Color Style: Discrete chunks
   Color Levels: 6


## 1. Import Required Libraries

In [96]:
import h5py
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from pathlib import Path

# Set plotly to work in notebooks
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)


## 2. Read H5 File Data

In [97]:
# Define available files and apply configuration-based selection
outputs_dir = Path("outputs")
h5_files = list(outputs_dir.glob("*.h5"))

print("Available HDF5 files:")
for i, file in enumerate(h5_files):
    with h5py.File(file, 'r') as f:
        n_combinations = f.attrs.get('total_combinations', 'unknown')
        test_mode = f.attrs.get('test_mode', 'unknown')
        file_size = file.stat().st_size / 1024**2  # MB
    print(f"{i+1}. {file.name} - {n_combinations} combinations ({test_mode}) - {file_size:.1f} MB")

# File selection based on configuration
if SELECTED_FILE is not None:
    # Use explicitly specified file
    selected_file_path = outputs_dir / SELECTED_FILE
    if selected_file_path.exists():
        selected_file = selected_file_path
        print(f"\n🎯 Using explicitly selected file: {SELECTED_FILE}")
    else:
        print(f"\n❌ Specified file '{SELECTED_FILE}' not found!")
        print(f"Available files: {[f.name for f in h5_files]}")
        raise FileNotFoundError(f"File '{SELECTED_FILE}' not found in outputs directory")
else:
    # Automatic selection based on preferences
    selected_file = None
    
    if PREFER_SMALL_TEST:
        # Look for small test files first
        small_test_files = [f for f in h5_files if 'small_test' in f.name]
        if small_test_files:
            if USE_MOST_RECENT:
                selected_file = max(small_test_files, key=lambda f: f.stat().st_mtime)
            else:
                selected_file = small_test_files[0]
            print(f"\n🧪 Using small test file: {selected_file.name}")
    
    if selected_file is None:
        # No small test files found or not preferred, use regular files
        if USE_MOST_RECENT:
            selected_file = max(h5_files, key=lambda f: f.stat().st_mtime)
        else:
            selected_file = h5_files[0]
        print(f"\n📊 Using {'most recent' if USE_MOST_RECENT else 'first available'} file: {selected_file.name}")

print(f"✅ Selected file: {selected_file.name}")

# Load data from HDF5 file with better error handling
with h5py.File(selected_file, 'r') as f:
    print(f"\nFile metadata:")
    for key, value in f.attrs.items():
        print(f"  {key}: {value}")
    
    print(f"\nAvailable datasets:")
    main_datasets = []
    other_items = []
    
    for key in f.keys():
        if isinstance(f[key], h5py.Dataset):
            shape = f[key].shape
            dtype = f[key].dtype
            print(f"  {key}: shape {shape}, dtype {dtype}")
            # Only include 1D datasets for main data
            if len(shape) == 1:
                main_datasets.append(key)
            else:
                other_items.append(key)
        else:
            print(f"  {key}: {type(f[key])} (group)")
            other_items.append(key)
    
    print(f"\nLoading main datasets: {len(main_datasets)} items")
    
    # Load only 1D datasets that are likely to be main result data
    data = {}
    expected_length = None
    
    for key in main_datasets:
        try:
            dataset_array = f[key][:]
            if expected_length is None:
                expected_length = len(dataset_array)
                print(f"Setting expected length from '{key}': {expected_length}")
            
            if len(dataset_array) == expected_length:
                data[key] = dataset_array
                print(f"  ✅ Loaded {key}: {len(dataset_array)} elements")
            else:
                print(f"  ⚠️  Skipped {key}: length mismatch ({len(dataset_array)} vs {expected_length})")
                
        except Exception as e:
            print(f"  ❌ Error loading {key}: {e}")
    
    # Handle groups (like parameter_fields) separately if needed
    if 'parameter_fields' in f:
        print(f"\nFound parameter_fields group:")
        param_group = f['parameter_fields']
        for subkey in param_group.keys():
            try:
                param_data = param_group[subkey][:]
                print(f"  {subkey}: {param_data.shape} - {param_data}")
            except Exception as e:
                print(f"  Error reading {subkey}: {e}")

print(f"\nSuccessfully loaded {len(data)} datasets")
if len(data) > 0:
    # Use the first dataset to determine the number of combinations
    first_key = list(data.keys())[0]
    n_combinations = len(data[first_key])
    print(f"Number of combinations: {n_combinations}")
    
    # Show a sample of what was loaded
    print(f"\nLoaded datasets:")
    for key, value in data.items():
        print(f"  {key}: length {len(value)}, dtype {value.dtype}")
        if len(value) > 0:
            print(f"    Sample: min={np.min(value):.3e}, max={np.max(value):.3e}")
else:
    print("❌ No valid datasets loaded!")
    raise ValueError("No data could be loaded from the HDF5 file")

Available HDF5 files:
1. dd_startup_results_test_20250916_132839.h5 - 1 combinations (small_test) - 0.1 MB
2. dd_startup_results_20250916_120903.h5 - 24300 combinations (unknown) - 0.6 MB
3. dd_startup_results_20250916_122212.h5 - 24300 combinations (unknown) - 0.6 MB

🎯 Using explicitly selected file: dd_startup_results_test_20250916_132839.h5
✅ Selected file: dd_startup_results_test_20250916_132839.h5

File metadata:
  computation_end_time: 1758022120.249694
  computation_start_time: 1758022119.0740252
  parameter_shapes: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  successful_startups: 1
  test_mode: small_test
  total_combinations: 1
  total_computation_time: 1.1756689548492432

Available datasets:
  Cost_per_kWh: shape (1,), dtype float64
  Dollar_Lost: shape (1,), dtype float64
  E_e_net_DD: shape (1,), dtype float64
  E_e_net_DT_full: shape (1,), dtype float64
  E_fusion_DT_full: shape (1,), dtype float64
  E_fusion_total_DD: shape (1,), dtype float64
  E_lost: shape (1,), dtype float64


## 3. Data Preprocessing and Filtering

In [98]:
# Configuration options have been moved to the top of the notebook
# Please see the "Configuration Options" section at the beginning to set:
# - SELECTED_FILE: Choose which HDF5 file to analyze
# - TARGET_VARIABLE: Choose target metric for analysis
# - Filter options and color settings

print("📝 Configuration options are now set at the top of the notebook")
print("🔧 Please modify the 'Configuration Options' cell if needed")
print(f"🎯 Current settings:")
print(f"   Target Variable: {TARGET_VARIABLE}")
print(f"   Filter Type: {FILTER_TYPE}")
print(f"   Filter Value: {FILTER_VALUE}")
print(f"   Color Style: {'Discrete chunks' if USE_DISCRETE_COLORS else 'Continuous gradient'}")
print(f"   Color Levels: {N_COLOR_CHUNKS}")

📝 Configuration options are now set at the top of the notebook
🔧 Please modify the 'Configuration Options' cell if needed
🎯 Current settings:
   Target Variable: Dollar_Lost
   Filter Type: economic
   Filter Value: 1000000000.0
   Color Style: Discrete chunks
   Color Levels: 6


In [99]:
# Convert to pandas DataFrame for easier manipulation
# First, check the data structure and handle different array lengths
print("Checking data structure:")
for key, value in data.items():
    if isinstance(value, np.ndarray):
        print(f"  {key}: shape {value.shape}, dtype {value.dtype}")
    else:
        print(f"  {key}: type {type(value)}")

# Filter out datasets that don't match the main data length
# Use linear_index as reference for the correct length
if 'linear_index' in data:
    expected_length = len(data['linear_index'])
    print(f"\nExpected data length (from linear_index): {expected_length}")
    
    # Filter data to only include arrays with the correct length
    filtered_data = {}
    for key, value in data.items():
        if isinstance(value, np.ndarray) and len(value) == expected_length:
            filtered_data[key] = value
        elif not isinstance(value, np.ndarray):
            print(f"Skipping non-array data: {key}")
        else:
            print(f"Skipping mismatched length data: {key} (length {len(value)})")
    
    print(f"\nFiltered data contains {len(filtered_data)} arrays with correct length")
    data_for_df = filtered_data
else:
    print("Warning: 'linear_index' not found. Using all data as-is.")
    data_for_df = data

# Create DataFrame from filtered data
try:
    df = pd.DataFrame(data_for_df)
    print(f"✅ Successfully created DataFrame with shape: {df.shape}")
except Exception as e:
    print(f"❌ Error creating DataFrame: {e}")
    # If still failing, let's try a more careful approach
    print("Attempting manual DataFrame creation...")
    
    # Find the most common array length
    lengths = []
    for key, value in data_for_df.items():
        if isinstance(value, np.ndarray):
            lengths.append(len(value))
    
    if lengths:
        most_common_length = max(set(lengths), key=lengths.count)
        print(f"Most common array length: {most_common_length}")
        
        # Use only arrays with the most common length
        final_data = {}
        for key, value in data_for_df.items():
            if isinstance(value, np.ndarray) and len(value) == most_common_length:
                final_data[key] = value
        
        df = pd.DataFrame(final_data)
        print(f"✅ Created DataFrame with shape: {df.shape}")
    else:
        raise ValueError("No valid arrays found for DataFrame creation")

print(f"Columns: {list(df.columns)}")

# Define input parameters (excluding calculated variables)
input_parameters = [
    'V_plasma',           # Plasma volume
    'T_i',               # Ion temperature
    'n_tot',             # Total density
    'tau_p_T',           # Tritium particle confinement time
    'tau_p_He3',         # He3 particle confinement time
    'P_aux',             # Auxiliary power (DD phase)
    'P_lost_rad',        # Radiated power loss (DD phase)
    'P_aux_all_DT',      # Auxiliary power (DT phase)
    'P_lost_rad_all_DT', # Radiated power loss (DT phase)
    'TBR_DT',            # Tritium breeding ratio (DT)
    'TBR_DDn',           # Tritium breeding ratio (DD neutron)
    'tau_ifc',           # In-facility cycle time
    'tau_ofc',           # Out-of-facility cycle time
    'eta_th',            # Thermal efficiency
    'plant_avail',       # Plant availability
    'Cost_per_kWh'       # Cost per kWh
]

# Target variable for coloring (use the configured target)
target_variable = TARGET_VARIABLE

# Variables to exclude from input parameters (but can be used as targets)
excluded_variables = [
    'linear_index',
    'injection_rate_max', 'sigmav_DT', 'sigmav_DD_p', 'sigmav_DD_n',  # Calculated from inputs
    'P_DT', 'P_DDn', 'P_DDp', 'P_DT_full',                          # ODE results
    'sol_success'                                                    # Solution status
]

# Available output metrics that can be used as target variables:
available_target_variables = [
    'Dollar_Lost',           # Economic metric [$]
    't_startup',            # Startup time [s]
    'Q_DD_total',           # Q factor for DD phase [-]
    'Q_DT_full_total',      # Q factor for DT phase [-]
    'P_fusion_DD_avg',      # Average DD fusion power [MW]
    'P_e_net_DD_avg',       # Average DD net electric power [MW]
    'P_e_net_DT_full_avg',  # Average DT net electric power [MW]
    'E_fusion_total_DD',    # Total DD fusion energy [MJ]
    'E_fusion_DT_full',     # Total DT fusion energy [MJ]
    'E_e_net_DD',           # Net DD electric energy [MJ]
    'E_e_net_DT_full',      # Net DT electric energy [MJ]
    'E_lost',               # Lost energy [MJ]
    'n_T_final'             # Final tritium density [m⁻³]
]

# Check which input parameters are actually available in the DataFrame
available_input_params = [param for param in input_parameters if param in df.columns]
missing_input_params = [param for param in input_parameters if param not in df.columns]

print(f"\nInput parameters analysis:")
print(f"  Available: {len(available_input_params)} / {len(input_parameters)}")
for param in available_input_params:
    print(f"    ✅ {param}")

if missing_input_params:
    print(f"  Missing: {len(missing_input_params)}")
    for param in missing_input_params:
        print(f"    ❌ {param}")

# Update input_parameters to only include available ones
input_parameters = available_input_params

# Check target variable
if target_variable not in df.columns:
    print(f"\n❌ Target variable '{target_variable}' not found in data!")
    print(f"Available columns: {list(df.columns)}")
    
    # Show available target options
    available_targets = [col for col in available_target_variables if col in df.columns]
    if available_targets:
        print(f"\n🎯 Available target variables:")
        for target in available_targets:
            print(f"    ✅ {target}")
        
        # Use the first available target as fallback
        target_variable = available_targets[0]
        print(f"\n🔄 Using fallback target variable: {target_variable}")
    else:
        print(f"\n❌ No suitable target variables found!")
        raise ValueError("No suitable target variable found")
else:
    print(f"✅ Target variable '{target_variable}' found")

# Show target variable info
if target_variable in df.columns:
    values = df[target_variable]
    finite_vals = values[np.isfinite(values)]
    if len(finite_vals) > 0:
        print(f"📊 {target_variable} statistics:")
        print(f"   Range: {finite_vals.min():.2e} to {finite_vals.max():.2e}")
        print(f"   Mean: {finite_vals.mean():.2e}")
        print(f"   Median: {finite_vals.median():.2e}")

print(f"\nFinal configuration:")
print(f"  Input parameters: {len(input_parameters)}")
print(f"  Target variable: {target_variable}")
print(f"  DataFrame shape: {df.shape}")

# Filter out infinite and invalid values
print(f"\nFiltering data...")
print(f"Before filtering: {len(df)} rows")

# Keep only finite target variable values
finite_mask = np.isfinite(df[target_variable])
df_filtered = df[finite_mask].copy()

print(f"After filtering infinite {target_variable}: {len(df_filtered)} rows")

# =============================================================================
# ECONOMIC FILTER: Maximum Dollar Lost threshold
# =============================================================================
# Set maximum Dollar_Lost threshold (None = no filter)
# You can adjust this value or set to None to include all scenarios

max_dollar_lost = None  # Set to a value (e.g., 1e8) to filter, or None for no filter
# max_dollar_lost = 1e8  # Example: Only scenarios with Dollar_Lost <= $100M

if max_dollar_lost is not None:
    print(f"\n🎯 Applying economic filter: {target_variable} <= ${max_dollar_lost:.2e}")
    economic_mask = df_filtered[target_variable] <= max_dollar_lost
    df_filtered = df_filtered[economic_mask].copy()
    print(f"After economic filtering: {len(df_filtered)} rows")
    print(f"Economic viability rate: {len(df_filtered)/len(df[finite_mask])*100:.1f}% of successful scenarios")
else:
    print(f"\n⚠️  No economic filter applied - showing all scenarios")

print(f"Success rate: {len(df_filtered)/len(df)*100:.1f}%")

# Display target variable distribution for reference
if len(df_filtered) > 0:
    target_stats = df_filtered[target_variable].describe()
    print(f"\n{target_variable} distribution in filtered data:")
    print(f"  Count:  {target_stats['count']:.0f}")
    print(f"  Min:    ${target_stats['min']:.2e}")
    print(f"  25%:    ${target_stats['25%']:.2e}")
    print(f"  Median: ${target_stats['50%']:.2e}")
    print(f"  75%:    ${target_stats['75%']:.2e}")
    print(f"  Max:    ${target_stats['max']:.2e}")
else:
    print(f"\n❌ No data remaining after filtering!")

print(f"\nFinal dataframe for plotting: {df_filtered.shape}")
print(f"Input parameters: {len(input_parameters)}")
if len(df_filtered) > 0:
    print(f"Target variable range: {df_filtered[target_variable].min():.2e} to {df_filtered[target_variable].max():.2e}")

# Show filter status for user reference
if max_dollar_lost is not None:
    print(f"\n📊 Economic Filter Status: ACTIVE (Max {target_variable} = ${max_dollar_lost:.2e})")
    print(f"   To disable filter: Set max_dollar_lost = None")
    print(f"   To adjust filter: Set max_dollar_lost = your_desired_value")
else:
    print(f"\n📊 Economic Filter Status: DISABLED")
    print(f"   To enable filter: Set max_dollar_lost = your_desired_threshold (e.g., 1e8)")
    print(f"   Example thresholds: 1e7 ($10M), 1e8 ($100M), 1e9 ($1B)")

Checking data structure:
  Cost_per_kWh: shape (1,), dtype float64
  Dollar_Lost: shape (1,), dtype float64
  E_e_net_DD: shape (1,), dtype float64
  E_e_net_DT_full: shape (1,), dtype float64
  E_fusion_DT_full: shape (1,), dtype float64
  E_fusion_total_DD: shape (1,), dtype float64
  E_lost: shape (1,), dtype float64
  P_DDn: shape (1,), dtype float64
  P_DDp: shape (1,), dtype float64
  P_DT: shape (1,), dtype float64
  P_DT_full: shape (1,), dtype float64
  P_aux: shape (1,), dtype float64
  P_aux_all_DT: shape (1,), dtype float64
  P_e_net_DD_avg: shape (1,), dtype float64
  P_e_net_DT_full_avg: shape (1,), dtype float64
  P_fusion_DD_avg: shape (1,), dtype float64
  P_lost_rad: shape (1,), dtype float64
  P_lost_rad_all_DT: shape (1,), dtype float64
  Q_DD_total: shape (1,), dtype float64
  Q_DT_full_total: shape (1,), dtype float64
  TBR_DDn: shape (1,), dtype float64
  TBR_DT: shape (1,), dtype float64
  T_i: shape (1,), dtype float64
  V_plasma: shape (1,), dtype float64
  et

## 4. Identify Input Variables and Target Variable

In [100]:
# Create readable labels for the plot
parameter_labels = {
    # Input parameters
    'V_plasma': 'Plasma Volume [m³]',
    'T_i': 'Ion Temperature [keV]',
    'n_tot': 'Total Density [m⁻³]',
    'tau_p_T': 'Tritium τp [s]',
    'tau_p_He3': 'He3 τp [s]',
    'P_aux': 'P_aux DD [MW]',
    'P_lost_rad': 'P_rad DD [MW]',
    'P_aux_all_DT': 'P_aux DT [MW]',
    'P_lost_rad_all_DT': 'P_rad DT [MW]',
    'TBR_DT': 'TBR DT [-]',
    'TBR_DDn': 'TBR DDn [-]',
    'tau_ifc': 'τ_ifc [h]',
    'tau_ofc': 'τ_ofc [h]',
    'eta_th': 'η_thermal [-]',
    'plant_avail': 'Plant Avail. [-]',
    'Cost_per_kWh': 'Cost [$/kWh]',
    
    # Output metrics (potential target variables)
    'Dollar_Lost': 'Dollar Lost [$]',
    't_startup': 'Startup Time [s]',
    'Q_DD_total': 'Q Factor DD [-]',
    'Q_DT_full_total': 'Q Factor DT [-]',
    'P_fusion_DD_avg': 'Avg DD Fusion Power [MW]',
    'P_e_net_DD_avg': 'Avg DD Net Power [MW]',
    'P_e_net_DT_full_avg': 'Avg DT Net Power [MW]',
    'E_fusion_total_DD': 'Total DD Fusion Energy [MJ]',
    'E_fusion_DT_full': 'Total DT Fusion Energy [MJ]',
    'E_e_net_DD': 'Net DD Electric Energy [MJ]',
    'E_e_net_DT_full': 'Net DT Electric Energy [MJ]',
    'E_lost': 'Lost Energy [MJ]',
    'n_T_final': 'Final Tritium Density [m⁻³]'
}

# Display parameter statistics
print("Parameter Statistics:")
print("=" * 80)
for param in input_parameters + [target_variable]:
    if param in df_filtered.columns:
        values = df_filtered[param]
        print(f"{parameter_labels.get(param, param):20s}: "
              f"min={values.min():8.2e}, max={values.max():8.2e}, "
              f"mean={values.mean():8.2e}, std={values.std():8.2e}")

# Check parameter ranges to ensure good visualization
print(f"\nParameter Ranges Check:")
for param in input_parameters:
    if param in df_filtered.columns:
        values = df_filtered[param]
        unique_vals = len(values.unique())
        range_ratio = (values.max() - values.min()) / values.std() if values.std() > 0 else 0
        print(f"{param:20s}: {unique_vals:3d} unique values, range/std = {range_ratio:.2f}")

# Create a subset for visualization if dataset is too large
max_points = 1000  # Limit for better performance
if len(df_filtered) > max_points:
    print(f"\nDataset has {len(df_filtered)} points. Sampling {max_points} for visualization...")
    df_plot = df_filtered.sample(n=max_points, random_state=42).copy()
else:
    df_plot = df_filtered.copy()

print(f"Using {len(df_plot)} points for visualization")

Parameter Statistics:
Plasma Volume [m³]  : min=1.50e+02, max=1.50e+02, mean=1.50e+02, std=     nan
Ion Temperature [keV]: min=1.70e+01, max=1.70e+01, mean=1.70e+01, std=     nan
Total Density [m⁻³] : min=1.70e+20, max=1.70e+20, mean=1.70e+20, std=     nan
Tritium τp [s]      : min=1.00e-01, max=1.00e-01, mean=1.00e-01, std=     nan
He3 τp [s]          : min=1.00e+00, max=1.00e+00, mean=1.00e+00, std=     nan
P_aux DD [MW]       : min=6.00e+07, max=6.00e+07, mean=6.00e+07, std=     nan
P_rad DD [MW]       : min=1.00e+07, max=1.00e+07, mean=1.00e+07, std=     nan
P_aux DT [MW]       : min=6.00e+07, max=6.00e+07, mean=6.00e+07, std=     nan
P_rad DT [MW]       : min=1.00e+07, max=1.00e+07, mean=1.00e+07, std=     nan
TBR DT [-]          : min=1.10e+00, max=1.10e+00, mean=1.10e+00, std=     nan
TBR DDn [-]         : min=7.00e-01, max=7.00e-01, mean=7.00e-01, std=     nan
τ_ifc [h]           : min=2.34e+04, max=2.34e+04, mean=2.34e+04, std=     nan
τ_ofc [h]           : min=4.50e+04, max=4

## 5. Create Parallel Coordinates Plot

In [101]:
# This cell is now integrated into the main plotting function above.
# Please use the configuration options in the "Customize Plot Colors and Styling" section.

print("📝 This functionality has been moved to the main plotting function.")
print("   Use the configuration options in the previous cell to customize your plot:")
print("   • ECONOMIC_FILTER: Set dollar threshold or None")
print("   • USE_DISCRETE_COLORS: True for chunks, False for gradient") 
print("   • N_COLOR_CHUNKS: Number of discrete color levels")
print("   Then re-run the previous cell to see your customized plot!")

📝 This functionality has been moved to the main plotting function.
   Use the configuration options in the previous cell to customize your plot:
   • ECONOMIC_FILTER: Set dollar threshold or None
   • USE_DISCRETE_COLORS: True for chunks, False for gradient
   • N_COLOR_CHUNKS: Number of discrete color levels
   Then re-run the previous cell to see your customized plot!


## 6. Customize Plot Colors and Styling

In [102]:
# Create the main plotting function with configurable target variables
def create_paracoords_plot(data_df, target_var=None, filter_value=None, filter_type='metric', use_discrete_colors=True, n_chunks=6):
    """
    Create parallel coordinates plot with configurable target variable and filtering
    
    Parameters:
    - data_df: DataFrame with all data
    - target_var: Target variable for coloring (e.g., 'Dollar_Lost', 't_startup')
    - filter_value: Filter threshold (None for no filter)
    - filter_type: 'economic' for Dollar_Lost, 'metric' for other variables
    - use_discrete_colors: If True, use discrete color chunks; if False, use gradient
    - n_chunks: Number of color chunks to create
    """
    
    # Use global target_variable if not specified
    if target_var is None:
        target_var = target_variable
    
    # Apply filtering
    df_filtered_plot = data_df[np.isfinite(data_df[target_var])].copy()
    
    # Apply threshold filter if specified
    if filter_value is not None:
        if filter_type == 'economic':
            # For economic metrics, filter for values <= threshold (lower is better)
            filter_mask = df_filtered_plot[target_var] <= filter_value
            filter_text = f"Economic Filter: ≤${filter_value:.1e}"
        else:
            # For other metrics, determine filter direction based on target variable
            if target_var in ['t_startup', 'Dollar_Lost', 'E_lost']:
                # For these metrics, lower is better
                filter_mask = df_filtered_plot[target_var] <= filter_value
                filter_text = f"Filter: {target_var} ≤ {filter_value:.2e}"
            else:
                # For Q factors, powers, energies, higher is typically better
                filter_mask = df_filtered_plot[target_var] >= filter_value
                filter_text = f"Filter: {target_var} ≥ {filter_value:.2e}"
        
        df_filtered_plot = df_filtered_plot[filter_mask].copy()
    else:
        filter_text = "No Filter"
    
    if len(df_filtered_plot) == 0:
        print(f"❌ No scenarios meet the filter criteria!")
        if filter_value is not None:
            print(f"   Filter: {filter_text}")
            print(f"   Try adjusting the threshold or set filter to None")
        return None, None, None  # Return tuple instead of just None
    
    # Sample data if too large
    max_points = 1000
    if len(df_filtered_plot) > max_points:
        df_plot_final = df_filtered_plot.sample(n=max_points, random_state=42).copy()
        sample_text = f"(sampled {max_points} points)"
    else:
        df_plot_final = df_filtered_plot.copy()
        sample_text = ""
    
    # Create discrete color chunks
    if use_discrete_colors:
        target_values = df_plot_final[target_var]
        
        # Define quantile-based chunks for better distribution
        quantiles = np.linspace(0, 1, n_chunks + 1)
        chunk_boundaries = target_values.quantile(quantiles).values
        
        # Create discrete color values and labels
        color_values = np.zeros(len(target_values))
        chunk_labels = []
        
        # Define color scheme based on target variable
        if target_var in ['t_startup', 'Dollar_Lost', 'E_lost']:
            # For "lower is better" metrics: Green (best) → Red (worst)
            colors = ['#2E8B57', '#32CD32', '#FFD700', '#FF8C00', '#FF4500', '#8B0000', '#000000']
            best_label = "Best"
            worst_label = "Worst"
        else:
            # For "higher is better" metrics: reverse the order
            colors = ['#8B0000', '#FF4500', '#FF8C00', '#FFD700', '#32CD32', '#2E8B57', '#000000']
            best_label = "Worst"
            worst_label = "Best"
        
        for i in range(n_chunks):
            if i == 0:
                mask = (target_values >= chunk_boundaries[i]) & (target_values <= chunk_boundaries[i+1])
                if target_var == 'Dollar_Lost':
                    label = f"{best_label}: ≤${chunk_boundaries[i+1]:.1e}"
                else:
                    label = f"{best_label}: ≤{chunk_boundaries[i+1]:.2e}"
            elif i == n_chunks - 1:
                mask = target_values > chunk_boundaries[i]
                if target_var == 'Dollar_Lost':
                    label = f"{worst_label}: >${chunk_boundaries[i]:.1e}"
                else:
                    label = f"{worst_label}: >{chunk_boundaries[i]:.2e}"
            else:
                mask = (target_values > chunk_boundaries[i]) & (target_values <= chunk_boundaries[i+1])
                if target_var == 'Dollar_Lost':
                    label = f"${chunk_boundaries[i]:.1e} - ${chunk_boundaries[i+1]:.1e}"
                else:
                    label = f"{chunk_boundaries[i]:.2e} - {chunk_boundaries[i+1]:.2e}"
            
            color_values[mask] = i
            chunk_labels.append(label)
        
        # Create custom discrete colorscale
        discrete_colorscale = []
        for i in range(len(colors)):
            if i < n_chunks:
                discrete_colorscale.extend([
                    [i/(n_chunks-1), colors[i]], 
                    [i/(n_chunks-1), colors[i]]
                ])
        
        colorscale = discrete_colorscale
        color_data = color_values
        
        print(f"📊 Created {n_chunks} discrete color chunks for {target_var}:")
        for i, label in enumerate(chunk_labels):
            count = np.sum(color_values == i)
            print(f"   {colors[i]} Chunk {i+1}: {label} ({count} scenarios)")
        
    else:
        # Use continuous gradient
        if target_var in ['t_startup', 'Dollar_Lost', 'E_lost']:
            colorscale = 'RdYlGn_r'  # Red (bad) to Green (good)
        else:
            colorscale = 'RdYlGn'    # Red (bad) to Green (good), but reversed
        color_data = df_plot_final[target_var]
    
    # Create dimensions for parallel coordinates
    dimensions = []
    for param in input_parameters:
        if param in df_plot_final.columns:
            values = df_plot_final[param]
            dimensions.append(
                dict(
                    label=parameter_labels.get(param, param),
                    values=values,
                    range=[values.min(), values.max()]
                )
            )
    
    # Add target variable as last dimension
    target_values_final = df_plot_final[target_var]
    dimensions.append(
        dict(
            label=parameter_labels.get(target_var, target_var),
            values=target_values_final,
            range=[target_values_final.min(), target_values_final.max()]
        )
    )
    
    # Create the plot
    fig = go.Figure(data=
        go.Parcoords(
            line=dict(
                color=color_data,
                colorscale=colorscale,
                showscale=True,
                colorbar=dict(
                    title=dict(
                        text=parameter_labels.get(target_var, target_var) if use_discrete_colors else parameter_labels.get(target_var, target_var),
                        font=dict(size=14)
                    ),
                    thickness=20,
                    len=0.8,
                    tickfont=dict(size=11),
                    # Custom tick labels for discrete colors
                    tickmode='array' if use_discrete_colors else 'linear',
                    tickvals=list(range(n_chunks)) if use_discrete_colors else None,
                    ticktext=[f"Level {i+1}" for i in range(n_chunks)] if use_discrete_colors else None
                ),
                cmin=0 if use_discrete_colors else target_values_final.min(),
                cmax=n_chunks-1 if use_discrete_colors else target_values_final.max()
            ),
            dimensions=dimensions
        )
    )
    
    # Calculate statistics
    success_rate = len(df_filtered_plot) / len(data_df[np.isfinite(data_df[target_var])]) * 100
    total_rate = len(df_filtered_plot) / len(data_df) * 100
    
    # Update layout
    color_type = "Discrete Color Chunks" if use_discrete_colors else "Gradient Colors"
    fig.update_layout(
        title=dict(
            text=f"DD Startup Parameter Sensitivity Analysis<br>"
                 f"<sub>{len(df_plot_final)} data points {sample_text} • "
                 f"Success rate: {total_rate:.1f}% • "
                 f"Target: {parameter_labels.get(target_var, target_var)} • "
                 f"{filter_text} • {color_type}</sub>",
            x=0.5,
            font=dict(size=18)
        ),
        font=dict(size=12),
        width=1400,
        height=700,
        margin=dict(l=100, r=120, t=120, b=100),
        paper_bgcolor='white',
        plot_bgcolor='white'
    )
    
    return fig, df_filtered_plot, chunk_labels if use_discrete_colors else None

# =============================================================================
# MAIN PLOTTING SECTION
# =============================================================================

print("🎨 Creating parallel coordinates plot...")
print(f"🎯 Target Variable: {TARGET_VARIABLE}")
print(f"📊 Filter: {FILTER_VALUE} ({FILTER_TYPE})")
print(f"🌈 Color scheme: {'Discrete chunks' if USE_DISCRETE_COLORS else 'Continuous gradient'}")

# Create the main plot
main_fig, filtered_data, color_chunks = create_paracoords_plot(
    df, 
    target_var=TARGET_VARIABLE,
    filter_value=FILTER_VALUE,
    filter_type=FILTER_TYPE,
    use_discrete_colors=USE_DISCRETE_COLORS,
    n_chunks=N_COLOR_CHUNKS
)

if main_fig is not None:
    main_fig.show()
    
    # Display summary statistics
    print(f"\n📈 Analysis Summary:")
    print(f"   • Total scenarios analyzed: {len(filtered_data)}")
    
    target_range_text = ""
    if TARGET_VARIABLE == 'Dollar_Lost':
        target_range_text = f"${filtered_data[TARGET_VARIABLE].min():.2e} to ${filtered_data[TARGET_VARIABLE].max():.2e}"
        median_text = f"${filtered_data[TARGET_VARIABLE].median():.2e}"
    else:
        target_range_text = f"{filtered_data[TARGET_VARIABLE].min():.2e} to {filtered_data[TARGET_VARIABLE].max():.2e}"
        median_text = f"{filtered_data[TARGET_VARIABLE].median():.2e}"
    
    print(f"   • {TARGET_VARIABLE} range: {target_range_text}")
    print(f"   • Median {TARGET_VARIABLE}: {median_text}")
    
    if USE_DISCRETE_COLORS and color_chunks:
        print(f"\n🎯 How to interpret colors:")
        if TARGET_VARIABLE in ['t_startup', 'Dollar_Lost', 'E_lost']:
            print(f"   🟢 Green shades: Best performance (lower values)")
            print(f"   🟡 Yellow/Orange: Moderate performance")
            print(f"   🔴 Red shades: Poor performance (higher values)")
        else:
            print(f"   🟢 Green shades: Best performance (higher values)")
            print(f"   🟡 Yellow/Orange: Moderate performance")
            print(f"   🔴 Red shades: Poor performance (lower values)")
    
    print(f"\n💡 Interactive Features:")
    print(f"   • Click and drag on any axis to filter parameter ranges")
    print(f"   • Focus on green lines for optimal parameter combinations")
    print(f"   • Use the color bar to understand performance levels")

else:
    print("❌ Unable to create plot. Check your filter settings.")
    # Show some helpful information about the data
    if len(df) > 0 and TARGET_VARIABLE in df.columns:
        target_vals = df[TARGET_VARIABLE][np.isfinite(df[TARGET_VARIABLE])]
        min_val = target_vals.min()
        max_val = target_vals.max()
        median_val = target_vals.median()
        
        print(f"\n📊 Available {TARGET_VARIABLE} range:")
        if TARGET_VARIABLE == 'Dollar_Lost':
            print(f"   • Minimum: ${min_val:.2e}")
            print(f"   • Median: ${median_val:.2e}")
            print(f"   • Maximum: ${max_val:.2e}")
            if FILTER_VALUE is not None:
                print(f"   • Current filter: ${FILTER_VALUE:.2e}")
        else:
            print(f"   • Minimum: {min_val:.2e}")
            print(f"   • Median: {median_val:.2e}")
            print(f"   • Maximum: {max_val:.2e}")
            if FILTER_VALUE is not None:
                print(f"   • Current filter: {FILTER_VALUE:.2e}")
        
        print(f"\n💡 Suggestions:")
        print(f"   • Try METRIC_FILTER = {median_val:.0e} (median)")
        print(f"   • Try METRIC_FILTER = {max_val:.0e} (maximum)")
        print(f"   • Or set METRIC_FILTER = None (no filter)")

# Quick configuration change section
print(f"\n⚙️  Quick Configuration Changes:")
print(f"   • Target variable: Change TARGET_VARIABLE (e.g., 't_startup', 'Q_DD_total')")
print(f"   • Economic filter: Change ECONOMIC_FILTER (for Dollar_Lost)")
print(f"   • Metric filter: Change METRIC_FILTER (for other targets)")
print(f"   • Color style: Toggle USE_DISCRETE_COLORS (True/False)")
print(f"   • Color levels: Adjust N_COLOR_CHUNKS (4-8 recommended)")
print(f"   • Then re-run this cell to update the plot!")

🎨 Creating parallel coordinates plot...
🎯 Target Variable: Dollar_Lost
📊 Filter: 1000000000.0 (economic)
🌈 Color scheme: Discrete chunks
❌ No scenarios meet the filter criteria!
   Filter: Economic Filter: ≤$1.0e+09
   Try adjusting the threshold or set filter to None
❌ Unable to create plot. Check your filter settings.

📊 Available Dollar_Lost range:
   • Minimum: $1.15e+09
   • Median: $1.15e+09
   • Maximum: $1.15e+09
   • Current filter: $1.00e+09

💡 Suggestions:
   • Try METRIC_FILTER = 1e+09 (median)
   • Try METRIC_FILTER = 1e+09 (maximum)
   • Or set METRIC_FILTER = None (no filter)

⚙️  Quick Configuration Changes:
   • Target variable: Change TARGET_VARIABLE (e.g., 't_startup', 'Q_DD_total')
   • Economic filter: Change ECONOMIC_FILTER (for Dollar_Lost)
   • Metric filter: Change METRIC_FILTER (for other targets)
   • Color style: Toggle USE_DISCRETE_COLORS (True/False)
   • Color levels: Adjust N_COLOR_CHUNKS (4-8 recommended)
   • Then re-run this cell to update the plot!


## Summary and Interpretation

The parallel coordinates plot above shows the relationship between all input parameters and your selected target metric. 

### How to interpret the plot:
- **Each line** represents one simulation run (parameter combination)
- **Line color** indicates the target variable performance:
  - For **"lower is better"** metrics (Dollar_Lost, t_startup, E_lost): 🟢 Green = Good performance (low values), 🔴 Red = Poor performance (high values)
  - For **"higher is better"** metrics (Q factors, powers): 🟢 Green = Good performance (high values), 🔴 Red = Poor performance (low values)
- **Interactive features**: Click and drag on any axis to filter data and focus on specific parameter ranges

### Key analysis questions:
1. **Which parameter combinations lead to the best performance?** (Look for green lines)
2. **Which parameters have the strongest influence on your target metric?** (Look for parameters where color changes dramatically)
3. **Are there parameter trade-offs?** (Look for crossing patterns between axes)
4. **What are the feasible parameter ranges for good performance?** (Focus on green line regions)

### Available Target Variables:
- **Dollar_Lost**: Economic performance metric
- **t_startup**: Time to reach DT conditions  
- **Q_DD_total, Q_DT_full_total**: Q factors for DD and DT phases
- **P_fusion_*_avg**: Average fusion powers
- **E_***: Energy metrics (fusion, net electric, lost)
- **n_T_final**: Final tritium density

### Next steps:
- Change TARGET_VARIABLE in the configuration to explore different metrics
- Use the interactive filtering to explore specific parameter ranges
- Identify optimal parameter combinations from the green lines
- Analyze parameter sensitivities and correlations
- Compare results across different target variables

## Interactive Target Variable and Filter Testing

Use this cell to quickly test different target variables and filter thresholds:

In [103]:
# =============================================================================
# QUICK TARGET VARIABLE AND FILTER TESTING
# =============================================================================
# Use this cell to quickly test different target variables and filters

# Target variable options (choose one):
test_target = 'Dollar_Lost'     # Economic metric
# test_target = 't_startup'       # Startup time [s]
# test_target = 'Q_DD_total'      # Q factor DD phase
# test_target = 'Q_DT_full_total' # Q factor DT phase
# test_target = 'P_fusion_DD_avg' # Average DD fusion power [MW]
# test_target = 'E_e_net_DD'      # Net DD electric energy [MJ]

# Filter options (choose appropriate values for your target):
test_filter = None           # No filter - show all scenarios

# Example filters for different targets:
# For Dollar_Lost:
# test_filter = 1e7          # ≤ $10M
# test_filter = 5e7          # ≤ $50M  
# test_filter = 1e8          # ≤ $100M

# For t_startup:
# test_filter = 1000         # ≤ 1000 seconds
# test_filter = 3600         # ≤ 1 hour

# For Q factors:
# test_filter = 0.1          # ≥ 0.1 (for Q factors, higher is better)
# test_filter = 0.01         # ≥ 0.01

# For powers/energies:
# test_filter = 10           # ≥ 10 MW or MJ (higher is better)
# test_filter = 50           # ≥ 50 MW or MJ

# Color scheme options:
use_chunks = True            # True for discrete color chunks, False for gradient
num_chunks = 6               # Number of color levels (4-8 recommended)

# Determine filter type
if test_target == 'Dollar_Lost':
    test_filter_type = 'economic'
else:
    test_filter_type = 'metric'

print(f"🔄 Testing configuration:")
print(f"   Target Variable: {test_target}")
print(f"   Filter Value: {test_filter}")
print(f"   Filter Type: {test_filter_type}")
print(f"   Color scheme: {'Discrete chunks' if use_chunks else 'Continuous gradient'}")
print(f"   Number of chunks: {num_chunks if use_chunks else 'N/A'}")

# Create test plot
test_fig, test_data, test_chunks = create_paracoords_plot(
    df,
    target_var=test_target,
    filter_value=test_filter,
    filter_type=test_filter_type,
    use_discrete_colors=use_chunks,
    n_chunks=num_chunks
)

if test_fig is not None:
    # Add "TEST" to the title
    current_title = test_fig.layout.title.text
    test_fig.update_layout(
        title=dict(
            text=current_title.replace("DD Startup Parameter", "🧪 TEST: DD Startup Parameter"),
            x=0.5,
            font=dict(size=18)
        )
    )
    
    test_fig.show()
    
    print(f"\n✅ Test plot created successfully!")
    print(f"   Target: {test_target}")
    print(f"   Scenarios shown: {len(test_data)}")
    
    if use_chunks and test_chunks:
        print(f"   Color chunks: {len(test_chunks)} levels")
    
    print(f"\n💡 If you like this configuration:")
    print(f"   1. Copy the settings to the main configuration section")
    print(f"   2. Update: TARGET_VARIABLE = '{test_target}'")
    if test_target == 'Dollar_Lost':
        print(f"   3. Update: ECONOMIC_FILTER = {test_filter}")
    else:
        print(f"   3. Update: METRIC_FILTER = {test_filter}")
    print(f"   4. Update: USE_DISCRETE_COLORS = {use_chunks}")
    print(f"   5. Update: N_COLOR_CHUNKS = {num_chunks}")
    print(f"   6. Re-run the main plotting cell")

else:
    print("❌ Test configuration failed. Try adjusting the filter threshold.")
    
    # Show target variable statistics for guidance
    if test_target in df.columns:
        target_vals = df[test_target][np.isfinite(df[test_target])]
        if len(target_vals) > 0:
            print(f"\n📊 {test_target} statistics:")
            print(f"   Min: {target_vals.min():.2e}")
            print(f"   Median: {target_vals.median():.2e}")
            print(f"   Max: {target_vals.max():.2e}")

print(f"\n🎨 Target Variable Guide:")
print(f"   • Dollar_Lost: Economic performance (lower = better)")
print(f"   • t_startup: Time to reach DT ignition (lower = better)")
print(f"   • Q_DD_total, Q_DT_full_total: Q factors (higher = better)")
print(f"   • P_fusion_*_avg: Fusion powers (higher = better)")
print(f"   • E_*: Energies (context dependent)")
print(f"   • n_T_final: Final tritium density (context dependent)")

print(f"\n💡 Filter Guidelines:")
print(f"   • 'Lower is better' metrics: filter shows values ≤ threshold")
print(f"   • 'Higher is better' metrics: filter shows values ≥ threshold")
print(f"   • Set filter = None to see all scenarios")

🔄 Testing configuration:
   Target Variable: Dollar_Lost
   Filter Value: None
   Filter Type: economic
   Color scheme: Discrete chunks
   Number of chunks: 6
📊 Created 6 discrete color chunks for Dollar_Lost:
   #2E8B57 Chunk 1: Best: ≤$1.2e+09 (1 scenarios)
   #32CD32 Chunk 2: $1.2e+09 - $1.2e+09 (0 scenarios)
   #FFD700 Chunk 3: $1.2e+09 - $1.2e+09 (0 scenarios)
   #FF8C00 Chunk 4: $1.2e+09 - $1.2e+09 (0 scenarios)
   #FF4500 Chunk 5: $1.2e+09 - $1.2e+09 (0 scenarios)
   #8B0000 Chunk 6: Worst: >$1.2e+09 (0 scenarios)



✅ Test plot created successfully!
   Target: Dollar_Lost
   Scenarios shown: 1
   Color chunks: 6 levels

💡 If you like this configuration:
   1. Copy the settings to the main configuration section
   2. Update: TARGET_VARIABLE = 'Dollar_Lost'
   3. Update: ECONOMIC_FILTER = None
   4. Update: USE_DISCRETE_COLORS = True
   5. Update: N_COLOR_CHUNKS = 6
   6. Re-run the main plotting cell

🎨 Target Variable Guide:
   • Dollar_Lost: Economic performance (lower = better)
   • t_startup: Time to reach DT ignition (lower = better)
   • Q_DD_total, Q_DT_full_total: Q factors (higher = better)
   • P_fusion_*_avg: Fusion powers (higher = better)
   • E_*: Energies (context dependent)
   • n_T_final: Final tritium density (context dependent)

💡 Filter Guidelines:
   • 'Lower is better' metrics: filter shows values ≤ threshold
   • 'Higher is better' metrics: filter shows values ≥ threshold
   • Set filter = None to see all scenarios


In [104]:
# Check available columns for target variables
print("Available columns in DataFrame:")
print("="*50)
for i, col in enumerate(df.columns, 1):
    values = df[col]
    if np.issubdtype(values.dtype, np.number):
        finite_vals = values[np.isfinite(values)]
        if len(finite_vals) > 0:
            print(f"{i:2d}. {col:25s} - {values.dtype} (min: {finite_vals.min():.2e}, max: {finite_vals.max():.2e})")
        else:
            print(f"{i:2d}. {col:25s} - {values.dtype} (all non-finite)")
    else:
        print(f"{i:2d}. {col:25s} - {values.dtype}")

# Identify potential target variables (output metrics)
print(f"\n🎯 Potential target variables (output metrics):")
output_candidates = []
for col in df.columns:
    if col not in input_parameters and col != 'linear_index':
        output_candidates.append(col)
        
for col in output_candidates:
    if col in df.columns:
        values = df[col]
        if np.issubdtype(values.dtype, np.number):
            finite_vals = values[np.isfinite(values)]
            if len(finite_vals) > 0:
                print(f"   ✅ {col:25s} - Range: {finite_vals.min():.2e} to {finite_vals.max():.2e}")
            else:
                print(f"   ❌ {col:25s} - All non-finite values")
        else:
            print(f"   ⚠️  {col:25s} - Non-numeric data")

Available columns in DataFrame:
 1. Cost_per_kWh              - float64 (min: 6.94e-08, max: 6.94e-08)
 2. Dollar_Lost               - float64 (min: 1.15e+09, max: 1.15e+09)
 3. E_e_net_DD                - float64 (min: 9.71e+15, max: 9.71e+15)
 4. E_e_net_DT_full           - float64 (min: 2.63e+16, max: 2.63e+16)
 5. E_fusion_DT_full          - float64 (min: 1.42e+17, max: 1.42e+17)
 6. E_fusion_total_DD         - float64 (min: 7.45e+16, max: 7.45e+16)
 7. E_lost                    - float64 (min: 1.66e+16, max: 1.66e+16)
 8. P_DDn                     - float64 (min: 1.56e+06, max: 1.56e+06)
 9. P_DDp                     - float64 (min: 1.69e+06, max: 1.69e+06)
10. P_DT                      - float64 (min: 5.40e+08, max: 5.40e+08)
11. P_DT_full                 - float64 (min: 1.04e+09, max: 1.04e+09)
12. P_aux                     - float64 (min: 6.00e+07, max: 6.00e+07)
13. P_aux_all_DT              - float64 (min: 6.00e+07, max: 6.00e+07)
14. P_e_net_DD_avg            - float64 (min: